# Malaika Voice Agent — Agentic IMCI Assessment on Colab

**Run on Colab with T4 GPU (free tier works).**

This notebook launches the **voice-first agentic IMCI assessment**:
1. Loads Gemma 4 E4B via Unsloth (vision + text)
2. Launches the FastAPI voice app with public tunnel
3. Open on your phone — speak, take photos, see skill cards + WHO classifications

**Features:**
- 12 clinical skills (vision, audio, speech parsing, classification)
- Belief state tracking (confirmed/uncertain/pending findings)
- Per-step WHO IMCI classification with severity badges (RED/YELLOW/GREEN)
- Sentence-level TTS streaming + filler audio
- Camera capture → vision skill → structured findings
- IMCI progress bar + skill execution cards in real-time

**Voice requires** `SMALLEST_API_KEY` (set in Colab secrets). Without it, text + image still work.

## Setup Checklist

Before running, add these to **Colab Secrets** (key icon in left sidebar):

| Secret | Required | Purpose |
|--------|----------|---------|
| `HF_TOKEN` | Yes | HuggingFace token for downloading Gemma 4 |
| `SMALLEST_API_KEY` | For voice | Real-time STT/TTS via Smallest AI |
| `NGROK_TOKEN` | Recommended | Stable public tunnel (free at ngrok.com) |

**Run order:** Cell 1 (installs + restarts) → Cell 2 (model load) → Cell 3 (launch)

In [ ]:
# Cell 1: Clone repo + install deps (kernel restarts at end)
!rm -rf /content/malaika
!git clone https://github.com/klickgenai/deepmind-hackathon.git /content/malaika
%cd /content/malaika

!pip install -q unsloth
!pip install -q --no-deps trl datasets librosa soundfile Pillow
!pip install -q structlog pyyaml httpx websockets pyngrok fastapi uvicorn

# Restart kernel to pick up unsloth patches
import os; os._exit(0)

In [ ]:
# Cell 2: Auth + Load Gemma 4 via Unsloth
# Run this AFTER kernel restarts from Cell 1

%cd /content/malaika
import sys, os, time
sys.path.insert(0, '/content/malaika')

# --- HuggingFace auth ---
from huggingface_hub import login
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("HF auth: Colab secrets")
except Exception:
    login()
    print("HF auth: manual")

# --- Smallest AI key (optional — for voice) ---
try:
    from google.colab import userdata
    smallest_key = userdata.get('SMALLEST_API_KEY')
    os.environ['SMALLEST_API_KEY'] = smallest_key
    print(f"Smallest AI: configured (voice enabled)")
except Exception:
    print("Smallest AI: not set (text + image only, no voice)")

# --- Load model ---
import torch
from unsloth import FastModel

print("\nLoading Gemma 4 E4B...")
t0 = time.time()
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
    full_finetuning=False,
)
print(f"Model loaded in {time.time()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("google/gemma-4-E4B-it")
print("Processor loaded")

# --- Quick vision sanity check ---
from PIL import Image
img = Image.new("RGB", (128, 128), color=(200, 150, 100))
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "What color is this? One word."}]}]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=input_text, images=[img], return_tensors="pt").to(model.device)
t0 = time.time()
out = model.generate(**inputs, max_new_tokens=10, do_sample=False, repetition_penalty=1.3)
resp = processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(f"Vision test: '{resp}' ({time.time()-t0:.1f}s)")
print("Ready!" if resp.strip() else "WARNING: Vision may not work")

In [ ]:
# Cell 3: Wire model into voice app + launch with public URL

from malaika.config import load_config
from malaika.voice_app import create_voice_app
from malaika.skills import SkillRegistry

config = load_config()
print(f"Skills registered: {len(SkillRegistry.list_all())}")

# Create the voice app with the loaded model
app = create_voice_app(model=model, processor=processor, config=config)
print("Voice app created")

# --- Launch with ngrok tunnel for public HTTPS URL ---
import threading
import uvicorn
from pyngrok import ngrok

# Set ngrok auth token (optional but recommended for stable tunnels)
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(ngrok_token)
    print("ngrok: authenticated")
except Exception:
    print("ngrok: no auth token (tunnel may be rate-limited)")

PORT = 8000

# Start uvicorn in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

import time; time.sleep(2)  # Wait for server to start

# Open ngrok tunnel
public_url = ngrok.connect(PORT, "http").public_url
# Force HTTPS
if public_url.startswith("http://"):
    public_url = public_url.replace("http://", "https://", 1)

print(f"\n{'='*60}")
print(f"  MALAIKA VOICE AGENT IS LIVE!")
print(f"{'='*60}")
print(f"\n  Open on your phone:")
print(f"  {public_url}")
print(f"\n  Features:")
has_voice = bool(os.environ.get('SMALLEST_API_KEY'))
if has_voice:
    print(f"  - Voice:  Tap the orb to talk")
print(f"  - Text:   Type messages in the input bar")
print(f"  - Camera: Tap camera icon to take photos")
print(f"  - Skills: Watch skill cards appear as AI analyzes")
print(f"  - IMCI:   See WHO classification cards per step")
print(f"\n{'='*60}")
print(f"  This cell stays running. Interrupt to stop.")
print(f"{'='*60}")

# Keep alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    ngrok.disconnect(public_url)
    print("\nShutdown complete.")